# Per-bucket z-scores for the fixed-space neutral shadows

Checkpoint (Docs/checkpoint 3 July 2026.md) §G / §I′ action **G2**.

The fixed-space neutral baselines — **eg** (729 ternary egene keys) and **dyn** (500 LUT input/output transition buckets) — previously exposed only an *aggregate* per-component-normalised excess scalar (`sim.eg_excess_pc()`, `sim.dyn_excess_pc()`). This notebook exercises the new **per-bucket z-scoring**, the well-posed "per-gene neutral shadow" instinct made rigorous in the fixed spaces:

$$z[k] = \frac{\text{observed cumulative activity}[k] - \text{shadow drift mean}[k]}{\sqrt{\text{closed-form WF drift variance}[k]}}$$

where the drift mean is the shadow's cumulative activity and the drift variance is the flux-gated Wright–Fisher form $m\,T^2 q_n(1-q_n)/N_p$ accumulated over steps.

Accessors: `sim.get_eg_zscores()` → float64[729], `sim.get_dyn_zscores()` → float64[500].

**Sanity claims tested below:**
1. Null/frozen arm (zero realised mutation flux): every bucket's z ≈ 0.
2. Selection arm: some buckets carry clearly large |z| — the real run concentrates cumulative activity on far fewer buckets than fitness-blind drift.

(Note: dyn z-scores are interpretable only where mu_lut>0.)

In [1]:
import os, sys, subprocess
import numpy as np

ROOT = os.path.abspath('.')
sys.path.insert(0, os.path.join(ROOT, 'python'))

# Build the C library if missing/stale (same recipe as tests/conftest.py).
src = os.path.join(ROOT, 'C', 'evoca.c')
lib = os.path.join(ROOT, 'C', 'libevoca.dylib')
if (not os.path.exists(lib)) or os.path.getmtime(lib) < os.path.getmtime(src):
    subprocess.run(['gcc', '-O2', '-Wall', '-fPIC', '-dynamiclib',
                    '-o', lib, src], cwd=ROOT, check=True)
    print('built libevoca.dylib')

from evoca_py import EvoCA, EGENE_KEY_COUNT, DYN_ACT_COUNT
print('EG buckets =', EGENE_KEY_COUNT, ' DYN buckets =', DYN_ACT_COUNT)

EG buckets = 729  DYN buckets = 500


In [2]:
def run(mu_lut, mu_egene, mu_egenome, tax, tax_per_egene,
        food_inc, gdiff, n_steps, N=40, seed=0):
    """Drive a short sim; return (alive, eg_zscores, dyn_zscores)."""
    s = EvoCA()
    s.init(N, food_inc=food_inc, m_scale=1.2, gdiff=gdiff, mu_lut=mu_lut,
           mu_egene=mu_egene, mu_egenome=mu_egenome, tax=tax,
           tax_per_egene=tax_per_egene)
    s.state(lut='gol', egenome='uniform', egenome_value=0b000011)
    rng = np.random.default_rng(seed)
    s.set_v(rng.integers(0, 2, (N, N), dtype=np.uint8))
    s.set_f_all(0.5); s.set_F_all(1.0); s.set_alive_fraction(0.6)
    for _ in range(n_steps):
        s.step()
    alive = int(np.ctypeslib.as_array(
        s._lib.evoca_get_alive(), shape=(N * N,)).sum())
    egz, dyz = s.get_eg_zscores(), s.get_dyn_zscores()
    s.free()
    return alive, egz, dyz

## Arm 1 — null / frozen (zero mutation flux, abundant uniform food)
With no realised flux the shadow tracks the realised composition exactly, so every per-bucket z must be ≈ 0.

In [3]:
alive_f, egz_f, dyz_f = run(
    mu_lut=0.0, mu_egene=0.0, mu_egenome=0.0,
    tax=0.002, tax_per_egene=0.0,
    food_inc=0.5, gdiff=0.0, n_steps=300, seed=3)
print(f'alive           = {alive_f}')
print(f'eg  |z| max     = {np.abs(egz_f).max():.3e}   (nonzero buckets: {int((egz_f != 0).sum())})')
print(f'dyn |z| max     = {np.abs(dyz_f).max():.3e}   (nonzero buckets: {int((dyz_f != 0).sum())})')
frozen_ok = (alive_f > 0 and np.abs(egz_f).max() < 1e-6
             and np.abs(dyz_f).max() < 1e-6)
print('frozen ~0 sanity:', 'PASS' if frozen_ok else 'FAIL')

alive           = 1600
eg  |z| max     = 0.000e+00   (nonzero buckets: 0)
dyn |z| max     = 0.000e+00   (nonzero buckets: 0)
frozen ~0 sanity: PASS


## Arm 2 — selection (scarce diffused food + per-egene tax, mutation on)
Eating/rule genomes are under real selection, so the run concentrates cumulative activity on few buckets. Some buckets must carry clearly large |z|.

In [4]:
alive_s, egz_s, dyz_s = run(
    mu_lut=0.005, mu_egene=0.004, mu_egenome=0.004,
    tax=0.02, tax_per_egene=0.006,
    food_inc=0.06, gdiff=0.5, n_steps=400, seed=3)
print(f'alive           = {alive_s}')
print(f'eg  |z| max     = {np.abs(egz_s).max():.2f}')
print(f'dyn |z| max     = {np.abs(dyz_s).max():.2f}')
print()
print('top 5 eg  z-scores (bucket -> z):')
for k in np.argsort(egz_s)[-5:][::-1]:
    print(f'   key {int(k):3d}  z = {egz_s[k]:8.2f}')
print('top 5 dyn z-scores (bucket -> z):')
for k in np.argsort(dyz_s)[-5:][::-1]:
    print(f'   transition {int(k):3d}  z = {dyz_s[k]:8.2f}')
sel_ok = (alive_s > 0 and np.abs(egz_s).max() > 3.0
          and np.abs(dyz_s).max() > 3.0
          and egz_s.max() > 2.0 and dyz_s.max() > 2.0)
print()
print('selection large-|z| sanity:', 'PASS' if sel_ok else 'FAIL')

alive           = 711
eg  |z| max     = 84.86
dyn |z| max     = 869.73

top 5 eg  z-scores (bucket -> z):
   key 368  z =    84.86
   key 362  z =    58.30
   key 360  z =    26.11
   key 365  z =    18.28
   key 351  z =    16.64
top 5 dyn z-scores (bucket -> z):
   transition   0  z =   869.73
   transition  10  z =    61.07
   transition   2  z =    55.64
   transition  50  z =    53.53
   transition   3  z =    28.87

selection large-|z| sanity: PASS


## Overall verdict

In [5]:
sep_ok = np.abs(dyz_s).max() > np.abs(dyz_f).max() + 3.0 and \
         np.abs(egz_s).max() > np.abs(egz_f).max() + 3.0
overall = frozen_ok and sel_ok and sep_ok
print('frozen ~0            :', 'PASS' if frozen_ok else 'FAIL')
print('selection large |z|  :', 'PASS' if sel_ok else 'FAIL')
print('selection >> frozen  :', 'PASS' if sep_ok else 'FAIL')
print('=' * 32)
print('OVERALL:', 'PASS' if overall else 'FAIL')
assert overall, 'per-bucket z-score sanity checks FAILED'

frozen ~0            : PASS
selection large |z|  : PASS
selection >> frozen  : PASS
OVERALL: PASS
